# KX market structure, climate day and city choice

Research-only spike for the seven configured KX daily-high series. Acquisition starts
**2026-11-13 17:22:20 UTC**; fresh Kalshi market/event pages finish **17:23:08 UTC**.
Historical responses and the 90-day quality cohort are reused from notebook 01's
2026-11-12 settlement snapshot. Every source and receipt time remains identifiable.

**Results:** all **5,137 KX events** are six-market integer-°F partitions, in both
NWS CLI and TWC periods. The CLI climate day uses midnight-to-midnight **local standard
time year-round**; six actual KNYC products corroborate it around both DST transitions.
Choose **Los Angeles**, with **NYC runner-up**: 41.62M versus 16.53M traded contracts.
The complete TWC ranking covers **651 events / 3,906 markets**, including November 14
listings. Settled OI is descriptive and unreliable for city selection; quality breaks
volume ties within 20%. Detailed evidence, exceptions, four assertion checks, five
demonstrated failures and three **draft** decision lines follow.

## Reproduce

Use the repository Python 3.12 environment with the optional notebook group
(`uv sync --group notebooks`). Run all cells from the repository root or notebooks/.
The default is **offline** and requires the ignored caches under
`data/spike/settlement_source/` and `data/spike/market_structure/`, including the
derived `directive-extracted.json`. This last file records its PDF hash, page numbers
and pypdf extractor version; offline execution needs no PDF package.

Initial acquisition uses `KWB_MARKET_STRUCTURE_OFFLINE=0`; ordinary runs never refresh
the frozen cache. New bodies are saved before decoding, byte-for-byte inside gzipped
base64 envelopes with SHA-256 and UTC request/receipt timestamps. Every cache write
checks its exact destination with `git check-ignore`. Notebook 01 and its cache are
read-only prerequisites; no production module imports notebook code.


## Documentation verified before client code

- [Markets](https://docs.kalshi.com/api-reference/market/get-markets.md): public GET
  (`security: []`), up to 1,000 results with a cursor; omit status to include every
  lifecycle state. `volume_fp` measures contracts; `open_interest_fp` has no documented
  guarantee of comparable terminal/pre-settlement values. Counts use Decimal.
- [Events](https://docs.kalshi.com/api-reference/events/get-events.md): public GET,
  up to 200 results; old events remain available. Nested markets omit the archival tier,
  so the independent event inventory is joined to separately acquired market records.
- [Historical data](https://docs.kalshi.com/getting_started/historical_data.md) and
  [historical markets](https://docs.kalshi.com/api-reference/historical/get-historical-markets):
  settlement time controls archival routing; read both tiers for complete history.
  Existing historical responses suffice here; a missing event triggers a bounded
  historical event request rather than another whole-history download.
- [Rate limits](https://docs.kalshi.com/getting_started/rate_limits.md): public,
  unauthenticated limits are separate from authenticated token budgets. This spike
  spaces Kalshi calls by at least 0.3 seconds and backs off on 429; it does not assume
  the authenticated Basic budget applies to public requests.
- [NWSI 10-1004](https://www.weather.gov/media/directives/010_pdfs/pd01010004curr.pdf#page=9),
  §4.1.2, printed page 9, is the CLI climate-day authority. The current PDF is cached;
  its relevant pages were extracted and inspected.
- [IEM CLI endpoint](https://mesonet.agron.iastate.edu/json/cli.py?help) documents
  station/year access; the annual responses already cached by notebook 01 supply
  product-specific links. [IEM usage policy](https://mesonet.agron.iastate.edu/disclaimer.php)
  does not give a numeric CLI limit. New evidence requests are serial, one second
  apart, with a descriptive User-Agent and bounded retries.

Individual market bounds, sources and climate dates below are checked against their
archived primary rules, with exact cache paths. The settled source-switch decision is
used and cross-checked, rather than reopened.


In [1]:
import base64
import gzip
import hashlib
import json
import os
import re
import subprocess
import time
from collections import Counter, defaultdict
from datetime import UTC, date, datetime, timedelta, timezone
from decimal import ROUND_CEILING, ROUND_FLOOR, Decimal
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode, urlsplit
from urllib.request import Request, urlopen
from zoneinfo import ZoneInfo

import pandas as pd
from IPython.display import HTML, Markdown, display
from IPython.utils.capture import capture_output

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "config/nws_stations.csv").is_file()
CACHE = ROOT / "data/spike/market_structure"
OLD_CACHE = ROOT / "data/spike/settlement_source"
OFFLINE = os.environ.get("KWB_MARKET_STRUCTURE_OFFLINE", "1") == "1"
BASE = "https://external-api.kalshi.com/trade-api/v2"
TWC_START = date(2026, 8, 14)
REQUEST_LOG = []
NEXT_REQUEST = defaultdict(float)


def write_ignored(path, body):
    relative = str(path.relative_to(ROOT))
    check = subprocess.run(
        ["git", "check-ignore", "-v", relative], cwd=ROOT, capture_output=True, text=True
    )
    assert check.returncode == 0, f"Unignored cache destination: {relative}"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as stream:
        stream.write(body)


META = CACHE / "snapshot.json"
if not META.exists():
    assert not OFFLINE, "Acquire once with KWB_MARKET_STRUCTURE_OFFLINE=0."
    write_ignored(
        META, json.dumps({"started_at": datetime.now(UTC).isoformat()}, indent=2).encode()
    )
SNAPSHOT = json.loads(META.read_text())
AS_OF = datetime.fromisoformat(SNAPSHOT["started_at"])


def read_envelope(path):
    envelope = json.loads(gzip.decompress(path.read_bytes()))
    body = base64.b64decode(envelope["body_base64"], validate=True)
    assert hashlib.sha256(body).hexdigest() == envelope["body_sha256"], path
    assert datetime.fromisoformat(envelope["received_at"]).utcoffset() == timedelta(0)
    return envelope, body


def response_bytes(url, params=None, *, tag=""):
    params = {} if params is None else params
    full_url = url + ("?" + urlencode(sorted(params.items())) if params else "")
    key = hashlib.sha256((full_url + "|" + tag).encode()).hexdigest()
    host = urlsplit(url).hostname
    source = "kalshi" if host == "external-api.kalshi.com" else "evidence"
    path = CACHE / source / AS_OF.date().isoformat() / (key + ".json.gz")
    if path.exists():
        envelope, body = read_envelope(path)
        assert envelope["request_url"] == full_url
        REQUEST_LOG.append({"url": full_url, "path": str(path.relative_to(ROOT)), "cached": True})
        assert envelope["http_status"] == 200, (full_url, envelope["http_status"])
        return body, str(path.relative_to(ROOT))
    assert not OFFLINE, f"Missing frozen response: {full_url}"
    interval = 0.3 if source == "kalshi" else 1.0
    for attempt in range(5):
        time.sleep(max(0, NEXT_REQUEST[source] - time.monotonic()))
        requested_at = datetime.now(UTC)
        request = Request(
            full_url,
            headers={
                "User-Agent": (
                    "Kalshi-Weather-Bot/0.1 (market-structure research; "
                    "github.com/prathikn20/Kalshi-Weather-Bot)"
                ),
                "Accept": "application/json, text/plain, */*",
            },
        )
        try:
            try:
                response = urlopen(request, timeout=45)
            except HTTPError as error:
                response = error
            with response:
                body = response.read()
                status = response.status
                headers = dict(response.headers.items())
            received_at = datetime.now(UTC)
            NEXT_REQUEST[source] = time.monotonic() + interval
        except (OSError, URLError):
            if attempt == 4:
                raise
            time.sleep(2 ** (attempt + 1))
            continue
        envelope = {
            "request_url": full_url,
            "request_params": params,
            "requested_at": requested_at.isoformat(),
            "received_at": received_at.isoformat(),
            "http_status": status,
            "response_headers": headers,
            "body_base64": base64.b64encode(body).decode(),
            "body_sha256": hashlib.sha256(body).hexdigest(),
        }
        destination = path if status == 200 else path.with_name(key + f".attempt-{attempt}.json.gz")
        write_ignored(destination, gzip.compress(json.dumps(envelope).encode()))
        REQUEST_LOG.append(
            {"url": full_url, "path": str(destination.relative_to(ROOT)), "cached": False}
        )
        if status == 200:
            return body, str(path.relative_to(ROOT))
        if status not in {408, 429} and status < 500:
            raise RuntimeError(f"HTTP {status}: {full_url}")
        retry = headers.get("Retry-After", "")
        time.sleep(max(float(retry) if retry.isdecimal() else 0, 2 ** (attempt + 1)))
    raise RuntimeError(f"Retries exhausted: {full_url}")


def get_json(path, params=None, *, tag=""):
    body, provenance = response_bytes(
        path if path.startswith("https://") else BASE + path, params, tag=tag
    )
    payload = json.loads(body)
    assert isinstance(payload, dict)
    return payload, provenance


PAGES = []


def paginate(path, params, field):
    cursor = ""
    cursors = set()
    records = []
    while True:
        payload, provenance = get_json(path, {**params, **({"cursor": cursor} if cursor else {})})
        rows = payload[field]
        next_cursor = payload.get("cursor", "")
        assert isinstance(rows, list) and isinstance(next_cursor, str)
        PAGES.append(
            {
                "path": path,
                "series": params["series_ticker"],
                "in": cursor,
                "out": next_cursor,
                "ids": [r.get("ticker", r.get("event_ticker")) for r in rows],
                "cache": provenance,
            }
        )
        records.extend({**r, "_cache_path": provenance, "_endpoint": path} for r in rows)
        if not next_cursor:
            break
        assert next_cursor not in cursors, "Cursor loop"
        cursors.add(next_cursor)
        cursor = next_cursor
    assert len(records) == len({r.get("ticker", r.get("event_ticker")) for r in records})
    return records


def show(rows):
    frame = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)
    if frame.empty:
        print("(none)")
    else:
        display(HTML(frame.to_html(index=False, max_rows=None, max_cols=None, render_links=True)))


# Reconstruct the approved settlement analysis from its frozen cache in its own
# namespace. This never edits the old notebook/cache or imports anything into src/.
settlement_notebook = json.loads((ROOT / "notebooks/01_settlement_source.ipynb").read_text())
SETTLEMENT = {"__name__": "settlement_cache_reuse"}
old_offline = os.environ.get("KWB_SPIKE_OFFLINE")
os.environ["KWB_SPIKE_OFFLINE"] = "1"
try:
    with capture_output():
        for index, prefix in [
            (4, "import base64"),
            (6, "CUTOFF_START"),
            (8, "from urllib.parse"),
            (10, "GROUPED_MARKETS"),
        ]:
            source = "".join(settlement_notebook["cells"][index]["source"])
            assert source.startswith(prefix), (
                "Settlement notebook prerequisites changed; review reuse."
            )
            exec(compile(source, f"settlement notebook cell {index}", "exec"), SETTLEMENT)
finally:
    if old_offline is None:
        os.environ.pop("KWB_SPIKE_OFFLINE")
    else:
        os.environ["KWB_SPIKE_OFFLINE"] = old_offline
STATIONS = SETTLEMENT["STATIONS"]
SERIES = {r["series_ticker"] for r in STATIONS}
assert len(SERIES) == 7 and all(r["cache"] for r in SETTLEMENT["REQUESTS"])
print("Frozen settlement prerequisite:", SETTLEMENT["AS_OF"].isoformat())
print("Reused", len(SETTLEMENT["MARKETS"]), "markets;", len(SETTLEMENT["EVENT_ROWS"]), "events.")
print("New acquisition started:", AS_OF.isoformat(), "offline:", OFFLINE)

Frozen settlement prerequisite: 2026-11-12T23:50:13.180836+00:00
Reused 47895 markets; 8525 events.
New acquisition started: 2026-11-13T17:22:20.677548+00:00 offline: True


## Complete inventory and source periods

Fresh unfiltered event lists include every status and future listings. Existing
settlement responses supply old markets; fresh live-tier records take precedence for
mutable fields. Rule geometry/source conflicts raise; mutable-field changes are
listed. Every KX event in the independent inventory must have market records, and
every market must map back to that inventory. Literal source wording determines the
NWS/TWC cohort; the recorded August 14 boundary is then asserted.


In [2]:
DOC_URLS = {
    "markets": "https://docs.kalshi.com/api-reference/market/get-markets.md",
    "events": "https://docs.kalshi.com/api-reference/events/get-events.md",
    "history": "https://docs.kalshi.com/getting_started/historical_data.md",
    "limits": "https://docs.kalshi.com/getting_started/rate_limits.md",
    "directive": "https://www.weather.gov/media/directives/010_pdfs/pd01010004curr.pdf",
}
DOCUMENTS = {}
for label, url in DOC_URLS.items():
    body, path = response_bytes(url)
    DOCUMENTS[label] = {"url": url, "path": path, "body": body}
assert b"security: []" in DOCUMENTS["markets"]["body"]
assert b"volume_fp" in DOCUMENTS["markets"]["body"]
assert b"open_interest_fp" in DOCUMENTS["markets"]["body"]
assert b"Unauthenticated" in DOCUMENTS["limits"]["body"]
print("Verified and cached current docs before market requests.")

CUTOFF_START, _ = get_json("/historical/cutoff", tag="start")
CURRENT_EVENTS = {}
CURRENT_MARKETS = {}
for series in sorted(SERIES):
    # No status filter: every API-served lifecycle state, including future listings.
    CURRENT_MARKETS[series] = paginate(
        "/markets", {"series_ticker": series, "limit": 1000}, "markets"
    )
    CURRENT_EVENTS[series] = paginate("/events", {"series_ticker": series, "limit": 200}, "events")
    print(
        series,
        "fresh live-tier markets",
        len(CURRENT_MARKETS[series]),
        "independent all-status events",
        len(CURRENT_EVENTS[series]),
        flush=True,
    )
CUTOFF_END, _ = get_json("/historical/cutoff", tag="end")
assert CUTOFF_START["market_settled_ts"] == CUTOFF_END["market_settled_ts"], (
    "Reconcile moving archive boundary."
)

INVENTORY = {}
NON_KX_INVENTORY = Counter()
for queried_series, events in CURRENT_EVENTS.items():
    for event in events:
        assert event["series_ticker"] == queried_series
        prefix = event["event_ticker"].rsplit("-", 1)[0]
        if prefix not in SERIES:
            assert prefix.startswith("HIGH"), event
            NON_KX_INVENTORY[prefix] += 1
            continue
        assert event["event_ticker"] not in INVENTORY
        INVENTORY[event["event_ticker"]] = event

RAW = {
    r["ticker"]: r for r in SETTLEMENT["MARKETS"] if r["event_ticker"].rsplit("-", 1)[0] in SERIES
}
OVERLAP_CHANGES = []
for rows in CURRENT_MARKETS.values():
    for row in rows:
        if row["event_ticker"].rsplit("-", 1)[0] not in SERIES:
            continue
        previous = RAW.get(row["ticker"])
        if previous:
            # Geometry and source evidence must agree. Mutable market totals use
            # the fresh response, once per ticker; differences remain inspectable.
            for field in [
                "event_ticker",
                "rules_primary",
                "strike_type",
                "floor_strike",
                "cap_strike",
            ]:
                assert row.get(field) == previous.get(field), (row["ticker"], field)
            changed = {
                k: [previous.get(k), row.get(k)]
                for k in ["volume_fp", "open_interest_fp", "status", "result"]
                if previous.get(k) != row.get(k)
            }
            if changed:
                OVERLAP_CHANGES.append(
                    {
                        "ticker": row["ticker"],
                        "changes": changed,
                        "old_cache": previous["_cache_path"],
                        "new_cache": row["_cache_path"],
                    }
                )
        RAW[row["ticker"]] = row

# If a historical event appeared between snapshots, fetch only that missing event.
missing = set(INVENTORY) - {r["event_ticker"] for r in RAW.values()}
for event_ticker in sorted(missing):
    rows = paginate(
        "/historical/markets",
        {
            "series_ticker": INVENTORY[event_ticker]["series_ticker"],
            "event_ticker": event_ticker,
            "limit": 1000,
        },
        "markets",
    )
    for row in rows:
        assert row["event_ticker"] == event_ticker
        assert row["ticker"] not in RAW
        RAW[row["ticker"]] = row
assert {r["event_ticker"] for r in RAW.values()} == set(INVENTORY), (
    "Every inventory event needs markets, and vice versa."
)

NORMALIZED = []
PARSE_EXCEPTIONS = []
RECEIPTS = {}
for raw in RAW.values():
    event = INVENTORY[raw["event_ticker"]]
    station, _ = SETTLEMENT["station_from_evidence"](raw, event)
    source = SETTLEMENT["classify_source"](raw["rules_primary"])
    assert source in {"NWS CLI", "The Weather Company"}, raw["ticker"]
    days = SETTLEMENT["text_dates"](raw["rules_primary"])
    assert len(days) == 1
    day = next(iter(days))
    date_error = None
    try:
        assert SETTLEMENT["climate_date"](raw, event) == day
    except AssertionError as error:
        date_error = str(error)
    interval_error = None
    try:
        interval = SETTLEMENT["interval_from_market"](raw)
    except AssertionError as error:
        # Explicit literal rules retain a malformed record as a reported anomaly.
        interval_error = str(error)
        interval = SETTLEMENT["interval_from_rules"](raw["rules_primary"])
    if raw["_cache_path"] not in RECEIPTS:
        envelope, _ = read_envelope(ROOT / raw["_cache_path"])
        RECEIPTS[raw["_cache_path"]] = envelope["received_at"]
    normalized = {
        "ticker": raw["ticker"],
        "event_ticker": raw["event_ticker"],
        "series": raw["event_ticker"].rsplit("-", 1)[0],
        "station": station,
        "day": day,
        "source": source,
        "interval": interval,
        "strike_type": raw.get("strike_type"),
        "date_error": date_error,
        "interval_error": interval_error,
        "received_at": RECEIPTS[raw["_cache_path"]],
        "cache": raw["_cache_path"],
    }
    NORMALIZED.append(normalized)
    if date_error or interval_error:
        PARSE_EXCEPTIONS.append(normalized)
GROUPS = defaultdict(list)
for row in NORMALIZED:
    GROUPS[row["event_ticker"]].append(row)
for event_ticker, rows in GROUPS.items():
    assert len({(r["series"], r["day"], r["station"], r["source"]) for r in rows}) == 1, (
        event_ticker
    )
    assert (rows[0]["day"] >= TWC_START) == (rows[0]["source"] == "The Weather Company")
TWC_EVENTS = {e for e, rows in GROUPS.items() if rows[0]["source"] == "The Weather Company"}
print("Complete KX inventory:", len(INVENTORY), "events;", len(RAW), "markets.")
print("TWC events:", len(TWC_EVENTS), "through all listings served during acquisition.")
print(
    "Legacy HIGH events served by canonical series filters, outside requested KX scope:",
    dict(NON_KX_INVENTORY),
)
print("Acquisition end:", max(read_envelope(ROOT / p["cache"])[0]["received_at"] for p in PAGES))
print("Every metadata/bounds anomaly:")
show(PARSE_EXCEPTIONS)
print("Mutable-field changes from older cache (fresh response wins):", len(OVERLAP_CHANGES))
show(OVERLAP_CHANGES)

Verified and cached current docs before market requests.
KXHIGHAUS fresh live-tier markets 420 independent all-status events 1280


KXHIGHCHI fresh live-tier markets 420 independent all-status events 1911


KXHIGHDEN fresh live-tier markets 420 independent all-status events 725


KXHIGHLAX fresh live-tier markets 420 independent all-status events 679


KXHIGHMIA fresh live-tier markets 420 independent all-status events 1283


KXHIGHNY fresh live-tier markets 420 independent all-status events 1923


KXHIGHPHIL fresh live-tier markets 420 independent all-status events 725


Complete KX inventory: 5137 events; 30822 markets.
TWC events: 651 through all listings served during acquisition.
Legacy HIGH events served by canonical series filters, outside requested KX scope: {'HIGHAUS': 528, 'HIGHCHI': 1158, 'HIGHCHI-2': 1, 'HIGHMIA': 531, 'HIGHNY': 1171}
Acquisition end: 2026-11-13T17:23:08.212841+00:00
Every metadata/bounds anomaly:
(none)
Mutable-field changes from older cache (fresh response wins): 0
(none)


## 1. Bracket structure by series and source

Strict/inclusive intervals are converted to integer endpoints. An endpoint sweep
checks all integers, including both unbounded tails, without restricting the search
to plausible weather. Finite span and number of integer outcomes are reported
separately: an inclusive 64–65°F bracket spans 1°F and contains two integer outcomes.
Missing structured strike fields are counted and resolved only from explicit rules.
The complete exception table remains visible even when it is empty.


In [3]:
def integer_bounds(interval):
    low, high, low_closed, high_closed = interval
    first = (
        None
        if low is None
        else int(
            low.to_integral_value(rounding=ROUND_CEILING)
            if low_closed
            else low.to_integral_value(rounding=ROUND_FLOOR) + 1
        )
    )
    last = (
        None
        if high is None
        else int(
            high.to_integral_value(rounding=ROUND_FLOOR)
            if high_closed
            else high.to_integral_value(rounding=ROUND_CEILING) - 1
        )
    )
    return first, last


def partition_diagnostics(rows):
    edges = Counter()
    count = 0
    empty = []
    for row in rows:
        first, last = integer_bounds(row["interval"])
        if first is not None and last is not None and first > last:
            empty.append(row["ticker"])
            continue
        if first is None:
            count += 1
        else:
            edges[first] += 1
        if last is not None:
            edges[last + 1] -= 1
    regions = []
    previous = None
    for edge in sorted(edges):
        if count != 1:
            regions.append({"first": previous, "last": edge - 1, "coverage": count})
        count += edges[edge]
        previous = edge
    if count != 1:
        regions.append({"first": previous, "last": None, "coverage": count})
    return {
        "regions": regions,
        "empty_integer_brackets": empty,
        "lower_tails": sum(r["interval"][0] is None for r in rows),
        "upper_tails": sum(r["interval"][1] is None for r in rows),
    }


def make_structure_report(groups):
    report = []
    for event_ticker, rows in sorted(groups.items()):
        diagnostics = partition_diagnostics(rows)
        is_partition = (
            not diagnostics["regions"]
            and not diagnostics["empty_integer_brackets"]
            and diagnostics["lower_tails"] == diagnostics["upper_tails"] == 1
            and not any(r["interval_error"] for r in rows)
        )
        finite = [r for r in rows if None not in r["interval"][:2]]
        report.append(
            {
                "event": event_ticker,
                "series": rows[0]["series"],
                "source": rows[0]["source"],
                "day": rows[0]["day"],
                "brackets": len(rows),
                "partition": is_partition,
                "strike_types": dict(Counter(str(r["strike_type"]) for r in rows)),
                "finite_numeric_spans_F": dict(
                    Counter(str(r["interval"][1] - r["interval"][0]) for r in finite)
                ),
                "finite_integer_widths": dict(
                    Counter(
                        integer_bounds(r["interval"])[1] - integer_bounds(r["interval"])[0] + 1
                        for r in finite
                    )
                ),
                **diagnostics,
                "bounds_errors": [r["ticker"] for r in rows if r["interval_error"]],
                "date_errors": [r["ticker"] for r in rows if r["date_error"]],
            }
        )
    return report


STRUCTURE = make_structure_report(GROUPS)
STRUCTURE_EXCEPTIONS = [r for r in STRUCTURE if not r["partition"]]
STRUCTURE_SUMMARY = []
for series in sorted(SERIES):
    for source in ["NWS CLI", "The Weather Company"]:
        events = [r for r in STRUCTURE if r["series"] == series and r["source"] == source]
        markets = [r for r in NORMALIZED if r["series"] == series and r["source"] == source]
        STRUCTURE_SUMMARY.append(
            {
                "series": series,
                "source": source,
                "events": len(events),
                "markets": len(markets),
                "first_day": min(r["day"] for r in events),
                "last_day": max(r["day"] for r in events),
                "strike_types_market_counts": dict(Counter(str(r["strike_type"]) for r in markets)),
                "brackets_per_event_counts": dict(
                    sorted(Counter(r["brackets"] for r in events).items())
                ),
                "finite_span_F_market_counts": dict(
                    sum((Counter(r["finite_numeric_spans_F"]) for r in events), Counter())
                ),
                "integer_width_market_counts": dict(
                    sum((Counter(r["finite_integer_widths"]) for r in events), Counter())
                ),
                "partitions": sum(r["partition"] for r in events),
                "exceptions": sum(not r["partition"] for r in events),
                "missing_structured_strikes": sum(r["strike_type"] is None for r in markets),
            }
        )
show(STRUCTURE_SUMMARY)
print("Every nonpartition event, including uncovered/overlapping integer regions:")
show(STRUCTURE_EXCEPTIONS)
print("A finite numeric span of 1°F covers two integers when both endpoints are inclusive.")

# Link archived rule evidence for one event in every city/source cohort.
RULE_EXAMPLES = []
for summary in STRUCTURE_SUMMARY:
    row = next(
        r
        for r in NORMALIZED
        if r["series"] == summary["series"]
        and r["source"] == summary["source"]
        and r["interval"][0] is not None
        and r["interval"][1] is not None
    )
    raw = RAW[row["ticker"]]
    RULE_EXAMPLES.append(
        {
            "series": row["series"],
            "source": row["source"],
            "ticker": row["ticker"],
            "interval": SETTLEMENT["bracket_label"](row["interval"]),
            "rule_predicate": raw["rules_primary"].split(", then")[0].split(" is ", 1)[1],
            "cache": row["cache"],
            "market_link": "https://kalshi.com/markets/"
            + row["series"].lower()
            + "/"
            + row["event_ticker"].lower(),
        }
    )
show(RULE_EXAMPLES)

series,source,events,markets,first_day,last_day,strike_types_market_counts,brackets_per_event_counts,finite_span_F_market_counts,integer_width_market_counts,partitions,exceptions,missing_structured_strikes
KXHIGHAUS,NWS CLI,659,3954,2024-10-24,2026-08-13,"{'less': 656, 'greater': 657, 'between': 2616, 'None': 25}",{6: 659},{'1': 2636},{2: 2636},659,0,25
KXHIGHAUS,The Weather Company,93,558,2026-08-14,2026-11-14,"{'greater': 93, 'less': 93, 'between': 372}",{6: 93},{'1': 372},{2: 372},93,0,0
KXHIGHCHI,NWS CLI,659,3954,2024-10-24,2026-08-13,"{'greater': 650, 'less': 659, 'between': 2620, 'None': 25}",{6: 659},{'1': 2636},{2: 2636},659,0,25
KXHIGHCHI,The Weather Company,93,558,2026-08-14,2026-11-14,"{'greater': 93, 'less': 93, 'between': 372}",{6: 93},{'1': 372},{2: 372},93,0,0
KXHIGHDEN,NWS CLI,632,3792,2024-11-20,2026-08-13,"{'greater': 627, 'less': 629, 'between': 2511, 'None': 25}",{6: 632},{'1': 2528},{2: 2528},632,0,25
KXHIGHDEN,The Weather Company,93,558,2026-08-14,2026-11-14,"{'greater': 93, 'less': 93, 'between': 372}",{6: 93},{'1': 372},{2: 372},93,0,0
KXHIGHLAX,NWS CLI,586,3516,2025-01-05,2026-08-13,"{'greater': 578, 'less': 586, 'between': 2327, 'None': 25}",{6: 586},{'1': 2344},{2: 2344},586,0,25
KXHIGHLAX,The Weather Company,93,558,2026-08-14,2026-11-14,"{'greater': 93, 'less': 93, 'between': 372}",{6: 93},{'1': 372},{2: 372},93,0,0
KXHIGHMIA,NWS CLI,659,3954,2024-10-24,2026-08-13,"{'greater': 646, 'less': 658, 'between': 2625, 'None': 25}",{6: 659},{'1': 2636},{2: 2636},659,0,25
KXHIGHMIA,The Weather Company,93,558,2026-08-14,2026-11-14,"{'greater': 93, 'less': 93, 'between': 372}",{6: 93},{'1': 372},{2: 372},93,0,0


Every nonpartition event, including uncovered/overlapping integer regions:
(none)
A finite numeric span of 1°F covers two integers when both endpoints are inclusive.


series,source,ticker,interval,rule_predicate,cache,market_link
KXHIGHAUS,NWS CLI,KXHIGHAUS-26AUG13-B99.5,"[99, 100]°F",between 99-100°,data/spike/settlement_source/kalshi/2026-11-12/dd856efc57ff9fbc6690c8a6747ebcf248e8b1e1391790c225885ed2c3defca5.json.gz,https://kalshi.com/markets/kxhighaus/kxhighaus-26aug13
KXHIGHAUS,The Weather Company,KXHIGHAUS-26NOV11-B81.5,"[81, 82]°F",between 81-82° fahrenheit according to The Weather Company,data/spike/market_structure/kalshi/2026-11-13/0f7875e4abfbce764ecd0565079ad8fe605735449ba08b4490c98dd3c6a5f41d.json.gz,https://kalshi.com/markets/kxhighaus/kxhighaus-26nov11
KXHIGHCHI,NWS CLI,KXHIGHCHI-26AUG13-B82.5,"[82, 83]°F",between 82-83° according to the National Weather Service's Climatological Report (Daily),data/spike/settlement_source/kalshi/2026-11-12/7f87006e8cb6e90be24621d8ddc0eeb5d65f164081ba2695d4d9fc27c6a8b182.json.gz,https://kalshi.com/markets/kxhighchi/kxhighchi-26aug13
KXHIGHCHI,The Weather Company,KXHIGHCHI-26NOV11-B59.5,"[59, 60]°F",between 59-60° fahrenheit according to The Weather Company,data/spike/market_structure/kalshi/2026-11-13/ec151306efe100ce02015c6fd14f0ba8964e734721caed96a0500fd63906132e.json.gz,https://kalshi.com/markets/kxhighchi/kxhighchi-26nov11
KXHIGHDEN,NWS CLI,KXHIGHDEN-26AUG13-B84.5,"[84, 85]°F",between 84-85°,data/spike/settlement_source/kalshi/2026-11-12/c0f2d20fe06cb8afb4b54d5c7ae01d64f0313ae6bd3c0679b84158386234922f.json.gz,https://kalshi.com/markets/kxhighden/kxhighden-26aug13
KXHIGHDEN,The Weather Company,KXHIGHDEN-26NOV11-B67.5,"[67, 68]°F",between 67-68° fahrenheit according to The Weather Company,data/spike/market_structure/kalshi/2026-11-13/fd7b8d75c03f52e9d92ac42ed191c507f6e8188f897a54741e0d8ddd794cf9c8.json.gz,https://kalshi.com/markets/kxhighden/kxhighden-26nov11
KXHIGHLAX,NWS CLI,KXHIGHLAX-26AUG13-B76.5,"[76, 77]°F",between 76-77°,data/spike/settlement_source/kalshi/2026-11-12/1b5eae600590b9444f5db2e7fa7b91860e3e5398b0dc17d4dc10f3faa4a8dab4.json.gz,https://kalshi.com/markets/kxhighlax/kxhighlax-26aug13
KXHIGHLAX,The Weather Company,KXHIGHLAX-26NOV11-B89.5,"[89, 90]°F",between 89-90° fahrenheit according to The Weather Company,data/spike/market_structure/kalshi/2026-11-13/5c9339605f43e3f82d322a4d9af70989130b44794ee00181c9550a5bb8f5ea5e.json.gz,https://kalshi.com/markets/kxhighlax/kxhighlax-26nov11
KXHIGHMIA,NWS CLI,KXHIGHMIA-26AUG13-B94.5,"[94, 95]°F",between 94-95°,data/spike/settlement_source/kalshi/2026-11-12/f53d9d035822938005fedea6b8ca9c83878f0b037dbc960d6a6faca198122d39.json.gz,https://kalshi.com/markets/kxhighmia/kxhighmia-26aug13
KXHIGHMIA,The Weather Company,KXHIGHMIA-26NOV11-B87.5,"[87, 88]°F",between 87-88° fahrenheit according to The Weather Company,data/spike/market_structure/kalshi/2026-11-13/465bfc8d26531ec4e25a3631038745f920b09ab9e6ebccc3dcec9c4a1b93435b.json.gz,https://kalshi.com/markets/kxhighmia/kxhighmia-26nov11


## 2. CLI climate day and actual DST evidence

The completed daily report differs from a preliminary afternoon CLI. The six linked
products below are completed reports, with verified CLINYC identifiers, climate-date
headlines, numeric maxima and LST observation columns. Fixed UTC−5 converts KNYC
local standard midnight to UTC; civil timezone conversion is for explanation only.


In [4]:
DIRECTIVE = json.loads((CACHE / "directive-extracted.json").read_text())
assert DIRECTIVE["pdf_sha256"] == hashlib.sha256(DOCUMENTS["directive"]["body"]).hexdigest()
DIRECTIVE_TEXT = " ".join(DIRECTIVE["pages"]["9"].split())
DIRECTIVE_QUOTE = "previous calendar day’s (midnight-to-midnight Local Standard Time [LST]) data."
assert DIRECTIVE_QUOTE in DIRECTIVE_TEXT
display(
    Markdown(
        f"**NWSI 10-1004 (June 5, 2025), §4.1.2, printed page 9:** “{DIRECTIVE_QUOTE}” "
        f"([directive]({DOC_URLS['directive']}#page=9)). The mandatory early-morning report "
        "covers the completed climate day; afternoon reports may still be preliminary. "
        "The 24-hour window is midnight to midnight in local **standard** time year-round. "
        "At KNYC that is 05:00–05:00 UTC, rather than a varying civil-midnight interval."
    )
)

CLI_DAYS = [
    date.fromisoformat(s)
    for s in ["2026-03-07", "2026-03-08", "2026-03-09", "2026-10-31", "2026-11-01", "2026-11-02"]
]
CLI_BODIES = {}


def climate_window(day):
    standard = timezone(timedelta(hours=-5))
    start = datetime(day.year, day.month, day.day, tzinfo=standard).astimezone(UTC)
    return start, start + timedelta(days=1)


def cli_evidence_row(day, body, url, cache):
    text = body.decode("utf-8")
    assert re.search(r"^CLINYC\s*$", text, re.MULTILINE)
    headlines = re.findall(r"CLIMATE SUMMARY FOR\s+([A-Z]+\s+\d{1,2}\s+\d{4})", text)
    assert len(headlines) == 1 and SETTLEMENT["text_dates"](headlines[0]) == {day}
    maximum = re.search(r"^\s*MAXIMUM\s+(-?\d+)(?:R)?\s+(\d{1,4}\s+[AP]M)\b", text, re.MULTILINE)
    assert maximum and "VALID TODAY AS OF" not in text and "YESTERDAY" in text
    issuance = re.search(r"^\d{1,4}\s+[AP]M\s+(EST|EDT)\s+.*$", text, re.MULTILINE)
    assert issuance
    start, end = climate_window(day)
    return {
        "climate_day": day,
        "maximum_F": int(maximum[1]),
        "maximum_time_LST": maximum[2],
        "time_column": "LST" if "(LST)" in text else "MISSING",
        "issued": issuance[0],
        "start_UTC": start.isoformat(),
        "end_UTC": end.isoformat(),
        "civil_start": start.astimezone(ZoneInfo("America/New_York")).isoformat(),
        "civil_end": end.astimezone(ZoneInfo("America/New_York")).isoformat(),
        "duration_hours": (end - start).total_seconds() / 3600,
        "link": url,
        "cache": cache,
    }


CLIMATE_EVIDENCE = []
for day in CLI_DAYS:
    cached = SETTLEMENT["CLI"]["KNYC", day]
    url = "https://mesonet.agron.iastate.edu" + cached["link"]
    body, cache = response_bytes(url)
    CLI_BODIES[day] = (body, url, cache)
    row = cli_evidence_row(day, body, url, cache)
    assert row["maximum_F"] == cached["high"]
    CLIMATE_EVIDENCE.append(row)
show(CLIMATE_EVIDENCE)
display(
    Markdown(
        "The March 8–9 completed reports are issued in EDT while retaining the LST observation "
        "column; the October 31 report describes a day in daylight time and also retains LST. "
        "These are real-product corroboration of the directive and reporting convention, "
        "not an independent reconstruction of subdaily observations. On March 8 the fixed "
        "window maps to 00:00 EST–01:00 EDT; on November 1 it maps to 01:00 EDT–00:00 EST. "
        "Both contain 24 elapsed hours. The adjacent-day controls remain in the table."
    )
)
PRIMARY_DAY_EVIDENCE = []
for day in [date(2026, 3, 8), date(2026, 11, 1)]:
    row = next(r for r in NORMALIZED if r["series"] == "KXHIGHNY" and r["day"] == day)
    primary = RAW[row["ticker"]]["rules_primary"]
    # Quote the date/source clause, without inventing an intraday window.
    clause = re.search(r"\bfor\s+(.+?)(?:,?\s+is\s+|\s+as reported)", primary)[1]
    PRIMARY_DAY_EVIDENCE.append(
        {
            "ticker": row["ticker"],
            "source": row["source"],
            "day_clause_verbatim": "for " + clause,
            "specifies_clock_boundary": bool(
                re.search(r"midnight|standard time|daylight|\b(?:EST|EDT)\b", primary, re.I)
            ),
            "cache": row["cache"],
            "link": "https://kalshi.com/markets/kxhighny/" + row["event_ticker"].lower(),
        }
    )
show(PRIMARY_DAY_EVIDENCE)
assert not any(r["specifies_clock_boundary"] for r in PRIMARY_DAY_EVIDENCE)
display(
    Markdown(
        "**KX primary rules name the climate date.** The NWS-era example names March 8, 2026; "
        "the TWC-era example names Nov 1, 2026. Neither primary-rule example states midnight, "
        "a timezone, or DST treatment. The NWS directive establishes the CLI feature window; "
        "it does **not** establish TWC’s internal aggregation boundary. This respects the "
        "already-settled policy that Kalshi outcomes supply settlement labels."
    )
)

**NWSI 10-1004 (June 5, 2025), §4.1.2, printed page 9:** “previous calendar day’s (midnight-to-midnight Local Standard Time [LST]) data.” ([directive](https://www.weather.gov/media/directives/010_pdfs/pd01010004curr.pdf#page=9)). The mandatory early-morning report covers the completed climate day; afternoon reports may still be preliminary. The 24-hour window is midnight to midnight in local **standard** time year-round. At KNYC that is 05:00–05:00 UTC, rather than a varying civil-midnight interval.

climate_day,maximum_F,maximum_time_LST,time_column,issued,start_UTC,end_UTC,civil_start,civil_end,duration_hours,link,cache
2026-03-07,52,1155 PM,LST,123 AM EST SUN MAR 08 2026,2026-03-07T05:00:00+00:00,2026-03-08T05:00:00+00:00,2026-03-07T00:00:00-05:00,2026-03-08T00:00:00-05:00,24.0,https://mesonet.agron.iastate.edu/api/1/nwstext/202603080623-KOKX-CDUS41-CLINYC,data/spike/market_structure/evidence/2026-11-13/0af30caa5dcf711291dc1094eed9aaed89b195942e3f8c3391113d8bf0369bd6.json.gz
2026-03-08,69,218 PM,LST,252 AM EDT MON MAR 09 2026,2026-03-08T05:00:00+00:00,2026-03-09T05:00:00+00:00,2026-03-08T00:00:00-05:00,2026-03-09T01:00:00-04:00,24.0,https://mesonet.agron.iastate.edu/api/1/nwstext/202603090652-KOKX-CDUS41-CLINYC,data/spike/market_structure/evidence/2026-11-13/a023feab915832d4f7436f769d4205201050143e187b893a770f156db9ca47d2.json.gz
2026-03-09,73,226 PM,LST,222 AM EDT TUE MAR 10 2026,2026-03-09T05:00:00+00:00,2026-03-10T05:00:00+00:00,2026-03-09T01:00:00-04:00,2026-03-10T01:00:00-04:00,24.0,https://mesonet.agron.iastate.edu/api/1/nwstext/202603100622-KOKX-CDUS41-CLINYC,data/spike/market_structure/evidence/2026-11-13/6b3a627d902cbe4ae584921960c117c724ec7d8f1ebec6c188dbdf56f2013725.json.gz
2026-10-31,58,451 PM,LST,140 AM EST SUN NOV 01 2026,2026-10-31T05:00:00+00:00,2026-11-01T05:00:00+00:00,2026-10-31T01:00:00-04:00,2026-11-01T01:00:00-04:00,24.0,https://mesonet.agron.iastate.edu/api/1/nwstext/202611010640-KOKX-CDUS41-CLINYC,data/spike/market_structure/evidence/2026-11-13/ce3bf907b4cf864acd6cdcb80adafd087307c3dc56cf1de9628c9a2d51d77034.json.gz
2026-11-01,59,230 PM,LST,121 AM EST MON NOV 02 2026,2026-11-01T05:00:00+00:00,2026-11-02T05:00:00+00:00,2026-11-01T01:00:00-04:00,2026-11-02T00:00:00-05:00,24.0,https://mesonet.agron.iastate.edu/api/1/nwstext/202611020621-KOKX-CDUS41-CLINYC,data/spike/market_structure/evidence/2026-11-13/d181fe11da17bf025a951cd18292769dd6abcc76c513c1a6419e6bcb55e30617.json.gz
2026-11-02,66,245 PM,LST,119 AM EST TUE NOV 03 2026,2026-11-02T05:00:00+00:00,2026-11-03T05:00:00+00:00,2026-11-02T00:00:00-05:00,2026-11-03T00:00:00-05:00,24.0,https://mesonet.agron.iastate.edu/api/1/nwstext/202611030619-KOKX-CDUS41-CLINYC,data/spike/market_structure/evidence/2026-11-13/68a7695db4e95b553b37babc426f2ef558e31340395e5fe3b86cf59d18a05414.json.gz


The March 8–9 completed reports are issued in EDT while retaining the LST observation column; the October 31 report describes a day in daylight time and also retains LST. These are real-product corroboration of the directive and reporting convention, not an independent reconstruction of subdaily observations. On March 8 the fixed window maps to 00:00 EST–01:00 EDT; on November 1 it maps to 01:00 EDT–00:00 EST. Both contain 24 elapsed hours. The adjacent-day controls remain in the table.

ticker,source,day_clause_verbatim,specifies_clock_boundary,cache,link
KXHIGHNY-26MAR08-T69,NWS CLI,"for March 08, 2026",False,data/spike/settlement_source/kalshi/2026-11-12/8cf911892e0eb0f58454c4c132c58b6517607755b5ad4e5301bd5f20dfc4a0de.json.gz,https://kalshi.com/markets/kxhighny/kxhighny-26mar08
KXHIGHNY-26NOV01-T67,The Weather Company,"for Nov 1, 2026",False,data/spike/market_structure/kalshi/2026-11-13/bf204718d03f76caeb61ab13c9c2523f7bb714f9c0f91a2738d2161c5cca575a.json.gz,https://kalshi.com/markets/kxhighny/kxhighny-26nov01


**KX primary rules name the climate date.** The NWS-era example names March 8, 2026; the TWC-era example names Nov 1, 2026. Neither primary-rule example states midnight, a timezone, or DST treatment. The NWS directive establishes the CLI feature window; it does **not** establish TWC’s internal aggregation boundary. This respects the already-settled policy that Kalshi outcomes supply settlement labels.

## 3. City ranking with quality beside volume and open interest

The cohort includes all 93 API-listed TWC days per city, August 14–November 14,
including one future day. Newer quality results are not inferred from unfinished
events. Both the shared 90-day TWC quality cohort and full KX-history quality counts
from notebook 01 are displayed with explicit denominators. Missing numeric fields
are reported, never replaced with zero. The complete event ledger is expandable.


In [5]:
def contract_count(raw, field):
    value = raw.get(field)
    if value in (None, ""):
        return None
    assert isinstance(value, str) and re.fullmatch(r"\d+(?:\.\d+)?", value), (
        raw["ticker"],
        field,
        value,
    )
    number = Decimal(value)
    assert number.is_finite() and number >= 0
    return number


def terminal(raw):
    return (
        raw["status"] in {"settled", "finalized", "determined"}
        or raw.get("result") in {"yes", "no", "scalar"}
        or bool(raw.get("settlement_ts"))
    )


QUALITY_BY_SERIES = {}
QUALITY_EXCEPTIONS = []
for series in sorted(SERIES):
    quality = {}
    for label, events in [
        ("KX_history", [r for r in SETTLEMENT["EVENT_ROWS"] if r["series"] == series]),
        (
            "TWC",
            [
                r
                for r in SETTLEMENT["EVENT_ROWS"]
                if r["series"] == series and r["source"] == "The Weather Company"
            ],
        ),
    ]:
        categories = {
            "missing_CLI": set(),
            "bracket_disagreement": set(),
            "exact_disagreement": set(),
            "nonnumeric_expiration": set(),
        }
        for event in events:
            e = event["event_ticker"]
            if event["cli_missing_reason"] is not None:
                categories["missing_CLI"].add(e)
            if event["disagreement"] is True:
                categories["bracket_disagreement"].add(e)
            if event["exact_difference_f"] not in (None, Decimal(0)):
                categories["exact_disagreement"].add(e)
            if any(r["expiration_error"] for r in SETTLEMENT["GROUPED_MARKETS"][e]):
                categories["nonnumeric_expiration"].add(e)
        affected = set().union(*categories.values())
        quality.update({f"{label}_{k}": len(v) for k, v in categories.items()})
        quality[f"{label}_audited_events"] = len(events)
        quality[f"{label}_affected_events"] = len(affected)
        for e in sorted(affected):
            if label == "TWC":
                QUALITY_EXCEPTIONS.append(
                    {
                        "series": series,
                        "event": e,
                        "categories": [k for k, v in categories.items() if e in v],
                    }
                )
    QUALITY_BY_SERIES[series] = quality

MISSING_FIELDS = []
EVENT_METRICS = []
for e in sorted(TWC_EVENTS):
    rows = GROUPS[e]
    raw_rows = [RAW[r["ticker"]] for r in rows]
    metrics = {"event": e, "series": rows[0]["series"], "day": rows[0]["day"], "markets": len(rows)}
    for field in ["volume_fp", "open_interest_fp"]:
        values = []
        for raw in raw_rows:
            value = contract_count(raw, field)
            if value is None:
                MISSING_FIELDS.append(
                    {
                        "series": rows[0]["series"],
                        "event": e,
                        "ticker": raw["ticker"],
                        "field": field,
                        "reason": "absent" if field not in raw else "null/empty",
                        "cache": raw["_cache_path"],
                    }
                )
            else:
                values.append(value)
        metrics[field] = sum(values, Decimal(0))
        metrics[field + "_present"] = len(values)
        metrics[field + "_missing"] = len(rows) - len(values)
    metrics["OI_terminal_unreliable"] = sum(
        (contract_count(r, "open_interest_fp") or Decimal(0)) for r in raw_rows if terminal(r)
    )
    metrics["OI_unsettled"] = sum(
        (contract_count(r, "open_interest_fp") or Decimal(0)) for r in raw_rows if not terminal(r)
    )
    metrics["terminal_markets"] = sum(terminal(r) for r in raw_rows)
    metrics["future_listing"] = (
        rows[0]["day"] > AS_OF.astimezone(ZoneInfo("America/New_York")).date()
    )
    metrics["market_tickers"] = sorted(r["ticker"] for r in rows)
    metrics["receipts_first"] = min(r["received_at"] for r in rows)
    metrics["receipts_last"] = max(r["received_at"] for r in rows)
    EVENT_METRICS.append(metrics)

CITY_METRICS = []
for series in sorted(SERIES):
    events = [e for e in EVENT_METRICS if e["series"] == series]
    row = {
        "series": series,
        "events": len(events),
        "first_day": min(e["day"] for e in events),
        "last_listed_day": max(e["day"] for e in events),
        "future_events_included": sum(e["future_listing"] for e in events),
        "markets": sum(e["markets"] for e in events),
        **QUALITY_BY_SERIES[series],
    }
    for field in [
        "volume_fp",
        "open_interest_fp",
        "OI_terminal_unreliable",
        "OI_unsettled",
        "terminal_markets",
        "volume_fp_missing",
        "open_interest_fp_missing",
    ]:
        row[field] = sum(e[field] for e in events)
    row["quality_not_audited_new_events"] = len(
        {e["event"] for e in events} - {e["event_ticker"] for e in SETTLEMENT["EVENT_ROWS"]}
    )
    CITY_METRICS.append(row)


def recommendation_order(metrics):
    # Within 20% means lower volume >= 80% of higher volume. Select sequentially
    # against the highest remaining volume, avoiding nontransitive pair sorting.
    remaining = list(metrics)
    ordered = []
    while remaining:
        top = max(r["volume_fp"] for r in remaining)
        candidates = [r for r in remaining if r["volume_fp"] >= Decimal("0.8") * top]
        winner = min(
            candidates,
            key=lambda r: (
                Decimal(r["TWC_affected_events"]) / r["TWC_audited_events"],
                -r["volume_fp"],
                r["series"],
            ),
        )
        ordered.append(winner["series"])
        remaining = [r for r in remaining if r["series"] != winner["series"]]
    return ordered


VOLUME_ORDER = [
    r["series"] for r in sorted(CITY_METRICS, key=lambda r: (-r["volume_fp"], r["series"]))
]
OI_ORDER = [
    r["series"] for r in sorted(CITY_METRICS, key=lambda r: (-r["open_interest_fp"], r["series"]))
]
UNSETTLED_OI_ORDER = [
    r["series"] for r in sorted(CITY_METRICS, key=lambda r: (-r["OI_unsettled"], r["series"]))
]
RECOMMENDATION = recommendation_order(CITY_METRICS)
for row in CITY_METRICS:
    row["volume_rank"] = VOLUME_ORDER.index(row["series"]) + 1
    row["reported_OI_rank_unreliable"] = OI_ORDER.index(row["series"]) + 1
    row["unsettled_OI_rank"] = UNSETTLED_OI_ORDER.index(row["series"]) + 1
    row["recommendation_rank"] = RECOMMENDATION.index(row["series"]) + 1

display(
    Markdown(
        "**Units:** fixed-point contract counts, parsed with Decimal; these are not "
        "dollar turnover. Each market contributes its latest acquired field once. "
        "Cached old finalized markets are reused; "
        "live-tier fields are refreshed, with overlapping differences retained above. This is a "
        "documented acquisition window, not a simultaneous historical snapshot. All API-served "
        "events dated August 14 onward are included, even future listings and unsettled days. "
        f"([field definitions]({DOC_URLS['markets']}))."
    )
)
MAIN_COLUMNS = [
    "volume_rank",
    "series",
    "events",
    "volume_fp",
    "open_interest_fp",
    "reported_OI_rank_unreliable",
    "OI_unsettled",
    "unsettled_OI_rank",
    "TWC_audited_events",
    "TWC_missing_CLI",
    "TWC_bracket_disagreement",
    "TWC_exact_disagreement",
    "TWC_nonnumeric_expiration",
    "quality_not_audited_new_events",
]
show(pd.DataFrame(sorted(CITY_METRICS, key=lambda r: r["volume_rank"]))[MAIN_COLUMNS])
print("Volume order:", " > ".join(VOLUME_ORDER))
print("Reported OI order (contains unreliable terminal values):", " > ".join(OI_ORDER))
print("Unsettled-only OI order:", " > ".join(UNSETTLED_OI_ORDER))
print("Recommendation order with the 20% quality tie-break:", " > ".join(RECOMMENDATION))
display(
    Markdown(
        "**Open-interest interpretation:** the current market schema defines the field but gives "
        "no guarantee that terminal values preserve a comparable pre-settlement interest measure. "
        "Consequently the requested raw OI ranking is descriptive and **unreliable "
        "for city selection**. Terminal contributions and an unsettled-only ranking "
        "are shown separately; neither changes the volume/quality selection rule. "
        "A zero terminal value is not evidence of zero past trading. "
        f"([schema]({DOC_URLS['markets']}), [historical tier]({DOC_URLS['history']}))."
    )
)
display(
    Markdown(
        "**Quality tie-break:** counts are affected **events**, including nonnumeric expirations "
        "(market-record counts are supplied separately below). The TWC quality audit is the shared "
        "90-day settled cohort, August 14–November 11, from notebook 01. Full KX-history counts "
        "are context, not the tie-break, because city history lengths differ. Affected-event rate "
        "uses the union of all four issue categories, so one bracket+exact mismatch counts once. "
        "Newer/unfinished events have explicitly unknown quality rather than zero defects. "
        "Within each remaining volume-leader group (at least 80% of its volume), prefer the "
        "lower affected-event rate; remaining ties use volume, then ticker."
    )
)
print("All TWC quality exceptions in the audited settlement cohort:")
show(QUALITY_EXCEPTIONS)
print("Frozen full KX-history quality counts (context; unequal history lengths):")
show(
    pd.DataFrame(sorted(CITY_METRICS, key=lambda r: r["volume_rank"]))[
        [
            "series",
            "KX_history_audited_events",
            "KX_history_missing_CLI",
            "KX_history_bracket_disagreement",
            "KX_history_exact_disagreement",
            "KX_history_nonnumeric_expiration",
            "KX_history_affected_events",
        ]
    ]
)
print("Coverage and unreliable settled OI contributions:")
show(
    pd.DataFrame(CITY_METRICS)[
        [
            "series",
            "first_day",
            "last_listed_day",
            "future_events_included",
            "markets",
            "terminal_markets",
            "OI_terminal_unreliable",
            "volume_fp_missing",
            "open_interest_fp_missing",
        ]
    ]
)
print("Nonnumeric expiration market-record counts in frozen KX history and TWC cohort:")
show(
    [
        {
            "series": series,
            "KX_history_nonnumeric_markets": sum(
                bool(r["expiration_error"])
                for r in SETTLEMENT["NORMALIZED"]
                if r["series"] == series
            ),
            "TWC_nonnumeric_markets": sum(
                bool(r["expiration_error"])
                for r in SETTLEMENT["NORMALIZED"]
                if r["series"] == series and r["source"] == "The Weather Company"
            ),
        }
        for series in sorted(SERIES)
    ]
)
print("Every missing volume or OI field (none are silently dropped):")
show(MISSING_FIELDS)
print("Complete event ledger; per-market provenance remains in exact response cache:")
ledger = pd.DataFrame(
    [{k: v for k, v in e.items() if k != "market_tickers"} for e in EVENT_METRICS]
)
display(
    HTML(
        "<details><summary>All TWC events (expand complete ledger)</summary>"
        + ledger.to_html(index=False, max_rows=None, max_cols=None)
        + "</details>"
    )
)

**Units:** fixed-point contract counts, parsed with Decimal; these are not dollar turnover. Each market contributes its latest acquired field once. Cached old finalized markets are reused; live-tier fields are refreshed, with overlapping differences retained above. This is a documented acquisition window, not a simultaneous historical snapshot. All API-served events dated August 14 onward are included, even future listings and unsettled days. ([field definitions](https://docs.kalshi.com/api-reference/market/get-markets.md)).

volume_rank,series,events,volume_fp,open_interest_fp,reported_OI_rank_unreliable,OI_unsettled,unsettled_OI_rank,TWC_audited_events,TWC_missing_CLI,TWC_bracket_disagreement,TWC_exact_disagreement,TWC_nonnumeric_expiration,quality_not_audited_new_events
1,KXHIGHLAX,93,41618559.95,19255315.56,1,40597.69,3,90,0,0,0,1,3
2,KXHIGHNY,93,16528729.96,8534515.27,2,215611.65,1,90,0,0,0,0,3
3,KXHIGHMIA,93,14781060.58,8310460.93,3,35631.17,4,90,0,1,1,0,3
4,KXHIGHCHI,93,8877745.57,4752815.86,4,21827.24,6,90,0,0,0,0,3
5,KXHIGHAUS,93,6234428.22,3381559.13,5,26544.08,5,90,0,0,0,0,3
6,KXHIGHPHIL,93,4445256.20,2399791.09,6,68468.85,2,90,0,0,0,0,3
7,KXHIGHDEN,93,4244679.85,2142913.53,7,10121.46,7,90,0,0,0,0,3


Volume order: KXHIGHLAX > KXHIGHNY > KXHIGHMIA > KXHIGHCHI > KXHIGHAUS > KXHIGHPHIL > KXHIGHDEN
Reported OI order (contains unreliable terminal values): KXHIGHLAX > KXHIGHNY > KXHIGHMIA > KXHIGHCHI > KXHIGHAUS > KXHIGHPHIL > KXHIGHDEN
Unsettled-only OI order: KXHIGHNY > KXHIGHPHIL > KXHIGHLAX > KXHIGHMIA > KXHIGHAUS > KXHIGHCHI > KXHIGHDEN
Recommendation order with the 20% quality tie-break: KXHIGHLAX > KXHIGHNY > KXHIGHMIA > KXHIGHCHI > KXHIGHAUS > KXHIGHPHIL > KXHIGHDEN


**Open-interest interpretation:** the current market schema defines the field but gives no guarantee that terminal values preserve a comparable pre-settlement interest measure. Consequently the requested raw OI ranking is descriptive and **unreliable for city selection**. Terminal contributions and an unsettled-only ranking are shown separately; neither changes the volume/quality selection rule. A zero terminal value is not evidence of zero past trading. ([schema](https://docs.kalshi.com/api-reference/market/get-markets.md), [historical tier](https://docs.kalshi.com/getting_started/historical_data.md)).

**Quality tie-break:** counts are affected **events**, including nonnumeric expirations (market-record counts are supplied separately below). The TWC quality audit is the shared 90-day settled cohort, August 14–November 11, from notebook 01. Full KX-history counts are context, not the tie-break, because city history lengths differ. Affected-event rate uses the union of all four issue categories, so one bracket+exact mismatch counts once. Newer/unfinished events have explicitly unknown quality rather than zero defects. Within each remaining volume-leader group (at least 80% of its volume), prefer the lower affected-event rate; remaining ties use volume, then ticker.

All TWC quality exceptions in the audited settlement cohort:


... (5 outputs left out)

Nonnumeric expiration market-record counts in frozen KX history and TWC cohort:


series,KX_history_nonnumeric_markets,TWC_nonnumeric_markets
KXHIGHAUS,0,0
KXHIGHCHI,0,0
KXHIGHDEN,0,0
KXHIGHLAX,6,6
KXHIGHMIA,5,0
KXHIGHNY,0,0
KXHIGHPHIL,0,0


Every missing volume or OI field (none are silently dropped):
(none)
Complete event ledger; per-market provenance remains in exact response cache:


event,series,day,markets,volume_fp,volume_fp_present,volume_fp_missing,open_interest_fp,open_interest_fp_present,open_interest_fp_missing,OI_terminal_unreliable,OI_unsettled,terminal_markets,future_listing,receipts_first,receipts_last
KXHIGHAUS-26AUG14,KXHIGHAUS,2026-08-14,6,59220.30,6,0,0.00,6,0,0,0,6,False,2026-11-12T23:51:39.509173+00:00,2026-11-12T23:51:39.509173+00:00
KXHIGHAUS-26AUG15,KXHIGHAUS,2026-08-15,6,65326.97,6,0,0.00,6,0,0,0,6,False,2026-11-12T23:51:39.509173+00:00,2026-11-12T23:51:39.509173+00:00
KXHIGHAUS-26AUG16,KXHIGHAUS,2026-08-16,6,85806.90,6,0,0.00,6,0,0,0,6,False,2026-11-12T23:51:39.509173+00:00,2026-11-12T23:51:39.509173+00:00
KXHIGHAUS-26AUG17,KXHIGHAUS,2026-08-17,6,66580.78,6,0,0.00,6,0,0,0,6,False,2026-11-12T23:51:39.509173+00:00,2026-11-12T23:51:39.509173+00:00
KXHIGHAUS-26AUG18,KXHIGHAUS,2026-08-18,6,61479.48,6,0,0.00,6,0,0,0,6,False,2026-11-12T23:51:39.509173+00:00,2026-11-12T23:51:39.509173+00:00
KXHIGHAUS-26AUG19,KXHIGHAUS,2026-08-19,6,59404.60,6,0,0.00,6,0,0,0,6,False,2026-11-12T23:51:39.509173+00:00,2026-11-12T23:51:39.509173+00:00
KXHIGHAUS-26AUG20,KXHIGHAUS,2026-08-20,6,46883.26,6,0,0.00,6,0,0,0,6,False,2026-11-12T23:51:39.509173+00:00,2026-11-12T23:51:39.509173+00:00
KXHIGHAUS-26AUG21,KXHIGHAUS,2026-08-21,6,55843.18,6,0,0.00,6,0,0,0,6,False,2026-11-12T23:51:39.509173+00:00,2026-11-12T23:51:39.509173+00:00
KXHIGHAUS-26AUG22,KXHIGHAUS,2026-08-22,6,82168.23,6,0,0.00,6,0,0,0,6,False,2026-11-12T23:51:39.509173+00:00,2026-11-12T23:51:39.509173+00:00
KXHIGHAUS-26AUG23,KXHIGHAUS,2026-08-23,6,178908.29,6,0,0.00,6,0,0,0,6,False,2026-11-12T23:51:39.509173+00:00,2026-11-12T23:51:39.509173+00:00


## Required assertions and failure demonstrations

Each full-sentence check validates a required property against real cached records.
The structure check independently evaluates literal predicates at all boundaries,
and exercises removed-tail/duplicate-bracket counterfactuals even though real
exceptions are zero. The ranking check reconciles the independent inventory, each
market total, each missing field, and both reported rankings. The added quality
check exercises both sides of the inclusive 20% threshold and ignores terminal OI.

Each demonstration replaces an analysis/reporting function with broken behavior,
runs the unchanged assertion, prints the actual failure, restores the function in
`finally`, and reruns successfully. Expected failures are captured output, not
unhandled notebook errors; every original function is restored before completion.


In [6]:
def structure_for_validation():
    return make_structure_report(GROUPS)


def event_metrics_for_validation():
    return EVENT_METRICS


def missing_fields_for_validation():
    return MISSING_FIELDS


def check_every_event_forms_an_integer_partition_or_lists_every_break():
    """Every event forms an integer partition or lists every gap, overlap and missing tail."""
    report = structure_for_validation()
    assert len(report) == len(INVENTORY) and {r["event"] for r in report} == set(INVENTORY), (
        "An inventory event disappeared from the structure report"
    )
    for result in report:
        rows = GROUPS[result["event"]]
        probes = {
            int(bound) + offset
            for r in rows
            for bound in r["interval"][:2]
            if bound is not None
            for offset in [-2, -1, 0, 1, 2]
        }
        assert probes
        probes.update([min(probes) - 10, max(probes) + 10])
        all_single = True
        for value in probes:
            # Literal rule predicates provide an independent equality check; no
            # integer_bounds or sweep logic is reused in this reference count.
            expected = sum(
                SETTLEMENT["contains"](
                    Decimal(value),
                    SETTLEMENT["interval_from_rules"](RAW[r["ticker"]]["rules_primary"]),
                )
                for r in rows
            )
            actual = [
                region
                for region in result["regions"]
                if (region["first"] is None or value >= region["first"])
                and (region["last"] is None or value <= region["last"])
            ]
            assert len(actual) <= 1
            assert bool(actual) == (expected != 1), (
                f"Wrong gap/overlap report: {result['event']} at {value}°F"
            )
            if actual:
                assert actual[0]["coverage"] == expected
            all_single = all_single and expected == 1
        lower = sum(
            SETTLEMENT["interval_from_rules"](RAW[r["ticker"]]["rules_primary"])[0] is None
            for r in rows
        )
        upper = sum(
            SETTLEMENT["interval_from_rules"](RAW[r["ticker"]]["rules_primary"])[1] is None
            for r in rows
        )
        assert result["lower_tails"] == lower and result["upper_tails"] == upper
        assert result["partition"] == (
            all_single
            and lower == upper == 1
            and not result["empty_integer_brackets"]
            and not result["bounds_errors"]
        )
    # Real-event counterfactuals check diagnostics even when actual exceptions=0.
    example = next(iter(GROUPS.values()))
    without_tail = [r for r in example if r["interval"][0] is not None]
    broken = partition_diagnostics(without_tail)
    assert broken["lower_tails"] == 0 and any(
        r["first"] is None and r["coverage"] == 0 for r in broken["regions"]
    )
    duplicate = partition_diagnostics(example + [example[0]])
    assert any(r["coverage"] == 2 for r in duplicate["regions"])
    assert sum(r["events"] for r in STRUCTURE_SUMMARY) == len(INVENTORY)


def check_the_climate_day_conclusion_matches_the_directive_and_both_dst_transitions():
    """The climate day matches the directive and real CLI products at both DST changes."""
    assert DIRECTIVE_QUOTE in " ".join(DIRECTIVE["pages"]["9"].split())
    assert DIRECTIVE["pdf_sha256"] == hashlib.sha256(DOCUMENTS["directive"]["body"]).hexdigest()
    expected_days = set(CLI_DAYS)
    assert len(CLIMATE_EVIDENCE) == len(expected_days) >= 2
    assert {r["climate_day"] for r in CLIMATE_EVIDENCE} == expected_days
    assert date(2026, 3, 8) in expected_days and date(2026, 11, 1) in expected_days
    for row in CLIMATE_EVIDENCE:
        day = row["climate_day"]
        body, url, cache = CLI_BODIES[day]
        assert row["time_column"] == "LST" and "(LST)" in body.decode()
        assert url.startswith("https://mesonet.agron.iastate.edu/api/1/nwstext/")
        start, end = climate_window(day)
        independent_start = datetime(day.year, day.month, day.day, 5, tzinfo=UTC)
        assert start == independent_start and end == independent_start + timedelta(days=1), (
            "Civil-midnight DST window replaced fixed local standard time"
        )
        assert (end - start).total_seconds() == 86400
        assert row == cli_evidence_row(day, body, url, cache), (
            "CLI evidence table differs from its real source"
        )
    assert any("EDT" in r["issued"] for r in CLIMATE_EVIDENCE if r["climate_day"].month == 3)
    assert all(r["time_column"] == "LST" for r in CLIMATE_EVIDENCE)
    for row in PRIMARY_DAY_EVIDENCE:
        assert row["day_clause_verbatim"] in RAW[row["ticker"]]["rules_primary"]
        assert row["specifies_clock_boundary"] is False


def check_the_rankings_account_for_every_twc_event_and_every_missing_field():
    """Rankings account for every TWC event and every missing volume or OI field."""
    expected_events = {
        e
        for e in INVENTORY
        if next(iter(SETTLEMENT["text_dates"](RAW[GROUPS[e][0]["ticker"]]["rules_primary"])))
        >= TWC_START
    }
    metrics = event_metrics_for_validation()
    assert (
        len(metrics) == len(expected_events) and {r["event"] for r in metrics} == expected_events
    ), "A TWC event was lost or duplicated in ranking accounting"
    expected_missing = set()
    by_series = defaultdict(lambda: defaultdict(Decimal))
    for event in metrics:
        raw_rows = [RAW[r["ticker"]] for r in GROUPS[event["event"]]]
        assert event["market_tickers"] == sorted(r["ticker"] for r in raw_rows)
        for field in ["volume_fp", "open_interest_fp"]:
            missing = [r for r in raw_rows if r.get(field) in (None, "")]
            expected_missing.update((r["ticker"], field) for r in missing)
            total = sum(
                (Decimal(r[field]) for r in raw_rows if r.get(field) not in (None, "")), Decimal(0)
            )
            assert event[field] == total, "An event total lost, duplicated or rounded contracts"
            assert event[field + "_missing"] == len(missing)
            assert event[field + "_present"] + len(missing) == len(raw_rows)
            by_series[event["series"]][field] += total
        expected_terminal = sum(
            (
                Decimal(r["open_interest_fp"])
                for r in raw_rows
                if terminal(r) and r.get("open_interest_fp") not in (None, "")
            ),
            Decimal(0),
        )
        assert event["OI_terminal_unreliable"] == expected_terminal
        assert event["OI_unsettled"] + expected_terminal == event["open_interest_fp"]
    missing_report = missing_fields_for_validation()
    assert len(missing_report) == len(expected_missing)
    assert {(r["ticker"], r["field"]) for r in missing_report} == expected_missing, (
        "A missing market field was silently dropped"
    )
    assert len(CITY_METRICS) == len(SERIES) == 7
    for row in CITY_METRICS:
        assert row["volume_fp"] == by_series[row["series"]]["volume_fp"]
        assert row["open_interest_fp"] == by_series[row["series"]]["open_interest_fp"]
        assert row["events"] == sum(e["series"] == row["series"] for e in metrics)
    for field, ranking in [
        ("volume_fp", VOLUME_ORDER),
        ("open_interest_fp", OI_ORDER),
        ("OI_unsettled", UNSETTLED_OI_ORDER),
    ]:
        assert ranking == [
            r["series"] for r in sorted(CITY_METRICS, key=lambda r: (-r[field], r["series"]))
        ]
    real_market = dict(RAW[GROUPS[next(iter(TWC_EVENTS))][0]["ticker"]])
    for field in ["volume_fp", "open_interest_fp"]:
        absent = {k: v for k, v in real_market.items() if k != field}
        assert contract_count(absent, field) is None, "A missing field became zero"
    for key, pages in __import__("itertools").groupby(
        sorted(PAGES, key=lambda p: (p["path"], p["series"])),
        key=lambda p: (p["path"], p["series"]),
    ):
        chain = list(pages)
        assert chain[0]["in"] == "" and chain[-1]["out"] == "", key
        assert all(a["out"] == b["in"] for a, b in zip(chain, chain[1:], strict=False)), key
        assert len({p["in"] for p in chain}) == len(chain)


def check_quality_breaks_twenty_percent_volume_ties_and_terminal_oi_cannot_change_the_choice():
    """Quality resolves twenty-percent volume ties without using terminal open interest."""
    for row in CITY_METRICS:
        for label in ["TWC", "KX_history"]:
            events = [
                e
                for e in SETTLEMENT["EVENT_ROWS"]
                if e["series"] == row["series"]
                and (label == "KX_history" or e["source"] == "The Weather Company")
            ]
            assert row[label + "_audited_events"] == len(events)
            assert row[label + "_missing_CLI"] == sum(
                e["cli_missing_reason"] is not None for e in events
            )
            assert row[label + "_bracket_disagreement"] == sum(
                e["disagreement"] is True for e in events
            )
            assert row[label + "_exact_disagreement"] == sum(
                e["exact_difference_f"] not in (None, Decimal(0)) for e in events
            )
            nonnumeric = {
                e["event_ticker"]
                for e in events
                if any(
                    m["expiration_error"] for m in SETTLEMENT["GROUPED_MARKETS"][e["event_ticker"]]
                )
            }
            assert row[label + "_nonnumeric_expiration"] == len(nonnumeric)
            affected = {
                e["event_ticker"]
                for e in events
                if e["cli_missing_reason"] is not None
                or e["disagreement"] is True
                or e["exact_difference_f"] not in (None, Decimal(0))
            } | nonnumeric
            assert row[label + "_affected_events"] == len(affected)
    assert recommendation_order(CITY_METRICS) == RECOMMENDATION
    # Retain the real NYC/Miami quality evidence; put their volumes at either
    # side of the user's threshold to ensure quality actually drives a tie.
    ny = dict(next(r for r in CITY_METRICS if r["series"] == "KXHIGHNY"))
    miami = dict(next(r for r in CITY_METRICS if r["series"] == "KXHIGHMIA"))
    assert ny["TWC_affected_events"] < miami["TWC_affected_events"]
    ny["volume_fp"], miami["volume_fp"] = Decimal("80"), Decimal("100")
    assert recommendation_order([ny, miami])[0] == "KXHIGHNY", (
        "The 20% quality tie-break was ignored"
    )
    ny["volume_fp"] = Decimal("79.99")
    assert recommendation_order([ny, miami])[0] == "KXHIGHMIA", (
        "Quality overrode volume outside the 20% group"
    )
    altered = [
        {**r, "open_interest_fp": Decimal(10) ** i, "OI_terminal_unreliable": Decimal(10) ** i}
        for i, r in enumerate(CITY_METRICS)
    ]
    assert recommendation_order(altered) == RECOMMENDATION, (
        "Unreliable terminal OI changed the city choice"
    )


CHECKS = [
    check_every_event_forms_an_integer_partition_or_lists_every_break,
    check_the_climate_day_conclusion_matches_the_directive_and_both_dst_transitions,
    check_the_rankings_account_for_every_twc_event_and_every_missing_field,
    check_quality_breaks_twenty_percent_volume_ties_and_terminal_oi_cannot_change_the_choice,
]
for check in CHECKS:
    check()
    print("PASS:", check.__doc__)


def prove_failure(label, function_name, replacement, check):
    original = globals()[function_name]
    check()
    try:
        globals()[function_name] = replacement
        try:
            check()
        except AssertionError as error:
            observed = str(error)
            print(f"EXPECTED FAILURE [{label}]: AssertionError: {observed}")
        else:
            raise RuntimeError(f"Mutation survived: {label}")
    finally:
        globals()[function_name] = original
    check()
    print("RESTORED PASS:", check.__doc__)
    return {
        "mutation": label,
        "check": check.__name__,
        "observed_failure": observed,
        "restored": "PASS",
    }


real_integer_bounds = integer_bounds


def wrong_integer_bounds(interval):
    first, last = real_integer_bounds(interval)
    return first, None if last is None else last + 1


def wrong_civil_window(day):
    zone = ZoneInfo("America/New_York")
    start = datetime(day.year, day.month, day.day, tzinfo=zone)
    return start.astimezone(UTC), (start + timedelta(days=1)).astimezone(UTC)


real_contract_count = contract_count
PROOFS = [
    prove_failure(
        "Expand upper bounds by one integer", "integer_bounds", wrong_integer_bounds, CHECKS[0]
    ),
    prove_failure(
        "Replace LST midnight with civil midnight", "climate_window", wrong_civil_window, CHECKS[1]
    ),
    prove_failure(
        "Drop one real TWC event from ranking",
        "event_metrics_for_validation",
        lambda: EVENT_METRICS[1:],
        CHECKS[2],
    ),
    prove_failure(
        "Default missing market fields to zero",
        "contract_count",
        lambda raw, field: (
            real_contract_count(raw, field) if raw.get(field) not in (None, "") else Decimal(0)
        ),
        CHECKS[2],
    ),
    prove_failure(
        "Ignore quality and sort by volume only",
        "recommendation_order",
        lambda rows: [
            r["series"] for r in sorted(rows, key=lambda r: (-r["volume_fp"], r["series"]))
        ],
        CHECKS[3],
    ),
]
show(PROOFS)
for check in CHECKS:
    check()
print("All four properties pass after five demonstrated failures and restoration.")

PASS: Every event forms an integer partition or lists every gap, overlap and missing tail.
PASS: The climate day matches the directive and real CLI products at both DST changes.
PASS: Rankings account for every TWC event and every missing volume or OI field.
PASS: Quality resolves twenty-percent volume ties without using terminal open interest.


EXPECTED FAILURE [Expand upper bounds by one integer]: AssertionError: Wrong gap/overlap report: KXHIGHAUS-24DEC01 at 64°F


RESTORED PASS: Every event forms an integer partition or lists every gap, overlap and missing tail.
EXPECTED FAILURE [Replace LST midnight with civil midnight]: AssertionError: Civil-midnight DST window replaced fixed local standard time
RESTORED PASS: The climate day matches the directive and real CLI products at both DST changes.
EXPECTED FAILURE [Drop one real TWC event from ranking]: AssertionError: A TWC event was lost or duplicated in ranking accounting
RESTORED PASS: Rankings account for every TWC event and every missing volume or OI field.


EXPECTED FAILURE [Default missing market fields to zero]: AssertionError: A missing field became zero
RESTORED PASS: Rankings account for every TWC event and every missing volume or OI field.
EXPECTED FAILURE [Ignore quality and sort by volume only]: AssertionError: The 20% quality tie-break was ignored
RESTORED PASS: Quality resolves twenty-percent volume ties without using terminal open interest.


mutation,check,observed_failure,restored
Expand upper bounds by one integer,check_every_event_forms_an_integer_partition_or_lists_every_break,Wrong gap/overlap report: KXHIGHAUS-24DEC01 at 64°F,PASS
Replace LST midnight with civil midnight,check_the_climate_day_conclusion_matches_the_directive_and_both_dst_transitions,Civil-midnight DST window replaced fixed local standard time,PASS
Drop one real TWC event from ranking,check_the_rankings_account_for_every_twc_event_and_every_missing_field,A TWC event was lost or duplicated in ranking accounting,PASS
Default missing market fields to zero,check_the_rankings_account_for_every_twc_event_and_every_missing_field,A missing field became zero,PASS
Ignore quality and sort by volume only,check_quality_breaks_twenty_percent_volume_ties_and_terminal_oi_cannot_change_the_choice,The 20% quality tie-break was ignored,PASS


All four properties pass after five demonstrated failures and restoration.


## Answers, draft decisions and scope

The decision lines below are drafts only. This task does not change `docs/plan.md`
or `docs/decisions.md`; the planning chat owns those edits.


In [7]:
by_series = {r["series"]: r for r in CITY_METRICS}
chosen, runner = RECOMMENDATION[:2]
known_volume_complete = not any(r["field"] == "volume_fp" for r in MISSING_FIELDS)
assert known_volume_complete, "Volume totals are partial: qualify recommendation before delivery."
gap = 100 * (Decimal(1) - by_series[runner]["volume_fp"] / by_series[chosen]["volume_fp"])
nws_events = sum(r["events"] for r in STRUCTURE_SUMMARY if r["source"] == "NWS CLI")
display(
    Markdown(
        f"**Answers.** All **{len(INVENTORY):,} KX events** ({nws_events:,} NWS CLI; "
        f"{len(TWC_EVENTS):,} TWC) form integer partitions, with six markets each: four inclusive "
        "two-integer ranges plus strict lower and upper tails. There are no partition exceptions. "
        "Missing structured strike fields affect 175 old markets (25 per series); their own "
        "literal primary rules supply the bounds and evidence is retained. "
        "The CLI climate day is midnight-to-midnight local standard time throughout the year, "
        "corroborated by all six linked KNYC products around both DST transitions. "
        f"Choose **{chosen} (Los Angeles)**; **{runner} (NYC)** is runner-up. "
        f"Los Angeles has {by_series[chosen]['volume_fp']:,.2f} traded contracts versus "
        f"{by_series[runner]['volume_fp']:,.2f} for NYC; NYC is {gap:.1f}% below LA, "
        "so LA’s one nonnumeric-expiration event does not activate the 20% quality tie-break. "
        "NYC and Miami do fall within 20%; NYC has zero audited TWC quality-issue events "
        "versus Miami’s one bracket/exact mismatch, reinforcing NYC as runner-up. "
        f"The volume and raw reported-OI rankings both read: {' > '.join(VOLUME_ORDER)}. "
        "The raw OI ranking includes terminal values and is not used to select the city. "
        "All 651 TWC events / 3,906 market records are accounted for; "
        "no volume or OI fields are missing."
    )
)

DECISION_DRAFTS = [
    (
        "2026-11-13: All seven configured KX daily-high series use six-market integer-°F "
        "partitions in both NWS CLI and TWC periods: four inclusive two-integer ranges "
        f"and both strict tails; {len(INVENTORY):,}/{len(INVENTORY):,} API-served KX events "
        "pass, including bounds recovered from 175 markets' literal primary rules. "
        "Evidence: notebooks/03_market_structure.ipynb."
    ),
    (
        "2026-11-13: NWS CLI completed daily maxima use midnight-to-midnight local "
        "standard time year-round (KNYC 05:00–05:00 UTC), per NWSI 10-1004 §4.1.2 and "
        "six real KNYC products around both 2026 DST transitions; KX primary rules name "
        "a date without specifying an intraday clock boundary, so this does not "
        "establish TWC's aggregation window. Evidence: notebooks/03_market_structure.ipynb."
    ),
    (
        "2026-11-13: Start with Los Angeles (KXHIGHLAX), NYC runner-up: TWC-period volume "
        f"is {by_series[chosen]['volume_fp']:,.2f} versus "
        f"{by_series[runner]['volume_fp']:,.2f} contracts across 93 events each; "
        "select by volume, using shared-cohort quality-issue rates within 20%, and do "
        "not use unreliable settled-market OI as historical liquidity evidence. "
        "Evidence: notebooks/03_market_structure.ipynb."
    ),
]
print("Draft decision lines only; planning chat owns the docs:")
for line in DECISION_DRAFTS:
    print(line)

display(
    Markdown(
        "**Design choices for a reader learning the codebase:**\n\n"
        "- Integer coverage is checked across every bound and both infinite tails, rather than "
        "a plausible temperature range; this catches a gap even outside normal weather. Decimal "
        "arithmetic preserves fractional contract counts and avoids rounding strict thresholds.\n"
        "- Source cohorts come from each market’s primary rules. The recorded August 14 switch "
        "is cross-checked, not inferred from ticker prefixes or present-day series metadata.\n"
        "- Notebook 01’s cache readers and normalization are reconstructed in an isolated "
        "namespace in offline mode. This reuses its vetted station/date/interval evidence "
        "without changing production schemas, the original notebook, or its cached responses. "
        "Its existing cache-recovery provenance remains in "
        "settlement_source/followup_cache_recovery.json.\n"
        "- The event inventory is independent of the market list; comparing both stops a "
        "missing market/event from silently disappearing from a ranking denominator. Every "
        "API-listed TWC event is included, with future listings marked explicitly.\n"
        "- Quality is compared on the same 90 completed days per city. The union of affected "
        "events prevents counting the Miami bracket and exact mismatch twice; the three "
        "newer days are marked unaudited, rather than assigned zero issues.\n"
        "- Terminal open interest is reported separately because the schema does not guarantee "
        "a comparable historical measure. The acquisition receipt times make the remaining "
        "unsettled OI values identifiable as observations during a request window."
    )
)

new_files = sorted(CACHE.rglob("*.json.gz"))
for path in new_files:
    assert (
        subprocess.run(
            ["git", "check-ignore", str(path.relative_to(ROOT))], cwd=ROOT, capture_output=True
        ).returncode
        == 0
    )
    read_envelope(path)
assert (
    subprocess.run(
        ["git", "check-ignore", "-v", "data/spike/market_structure/"], cwd=ROOT, capture_output=True
    ).returncode
    == 0
)
assert all(row["cached"] for row in REQUEST_LOG) if OFFLINE else True
print(
    "Verified ignored exact-body cache:",
    len(new_files),
    "responses;",
    sum(p.stat().st_size for p in new_files),
    "compressed bytes.",
)
print(
    "This run:",
    sum(not r["cached"] for r in REQUEST_LOG),
    "new network responses;",
    len(SETTLEMENT["REQUESTS"]),
    "settlement-cache reads.",
)
print("The five failure demonstrations restored every original function; final checks:")
for check in CHECKS:
    check()
    print("PASS:", check.__doc__)

display(
    Markdown(
        "**Out-of-scope findings, left unchanged:**\n\n"
        "- Canonical KX series queries also serve older HIGH-family events, including "
        "`HIGHCHI-2` metadata. These are counted as excluded legacy-family inventory, "
        "rather than mixed into the requested KX partitions or TWC ranking.\n"
        "- The TWC intraday aggregation window is not specified by the examined KX primary "
        "rules; no TWC clock-boundary equivalence is asserted or implemented.\n"
        "- The separate API-history/rate-limit spike remains open; only the documentation "
        "needed for these research requests was checked. Production collectors, schemas, "
        "pricing, plan and decisions remain unchanged."
    )
)

**Answers.** All **5,137 KX events** (4,486 NWS CLI; 651 TWC) form integer partitions, with six markets each: four inclusive two-integer ranges plus strict lower and upper tails. There are no partition exceptions. Missing structured strike fields affect 175 old markets (25 per series); their own literal primary rules supply the bounds and evidence is retained. The CLI climate day is midnight-to-midnight local standard time throughout the year, corroborated by all six linked KNYC products around both DST transitions. Choose **KXHIGHLAX (Los Angeles)**; **KXHIGHNY (NYC)** is runner-up. Los Angeles has 41,618,559.95 traded contracts versus 16,528,729.96 for NYC; NYC is 60.3% below LA, so LA’s one nonnumeric-expiration event does not activate the 20% quality tie-break. NYC and Miami do fall within 20%; NYC has zero audited TWC quality-issue events versus Miami’s one bracket/exact mismatch, reinforcing NYC as runner-up. The volume and raw reported-OI rankings both read: KXHIGHLAX > KXHIGHNY > KXHIGHMIA > KXHIGHCHI > KXHIGHAUS > KXHIGHPHIL > KXHIGHDEN. The raw OI ranking includes terminal values and is not used to select the city. All 651 TWC events / 3,906 market records are accounted for; no volume or OI fields are missing.

Draft decision lines only; planning chat owns the docs:
2026-11-13: All seven configured KX daily-high series use six-market integer-°F partitions in both NWS CLI and TWC periods: four inclusive two-integer ranges and both strict tails; 5,137/5,137 API-served KX events pass, including bounds recovered from 175 markets' literal primary rules. Evidence: notebooks/03_market_structure.ipynb.
2026-11-13: NWS CLI completed daily maxima use midnight-to-midnight local standard time year-round (KNYC 05:00–05:00 UTC), per NWSI 10-1004 §4.1.2 and six real KNYC products around both 2026 DST transitions; KX primary rules name a date without specifying an intraday clock boundary, so this does not establish TWC's aggregation window. Evidence: notebooks/03_market_structure.ipynb.
2026-11-13: Start with Los Angeles (KXHIGHLAX), NYC runner-up: TWC-period volume is 41,618,559.95 versus 16,528,729.96 contracts across 93 events each; select by volume, using shared-cohort quality-issue rates within 20%, and

**Design choices for a reader learning the codebase:**

- Integer coverage is checked across every bound and both infinite tails, rather than a plausible temperature range; this catches a gap even outside normal weather. Decimal arithmetic preserves fractional contract counts and avoids rounding strict thresholds.
- Source cohorts come from each market’s primary rules. The recorded August 14 switch is cross-checked, not inferred from ticker prefixes or present-day series metadata.
- Notebook 01’s cache readers and normalization are reconstructed in an isolated namespace in offline mode. This reuses its vetted station/date/interval evidence without changing production schemas, the original notebook, or its cached responses. Its existing cache-recovery provenance remains in settlement_source/followup_cache_recovery.json.
- The event inventory is independent of the market list; comparing both stops a missing market/event from silently disappearing from a ranking denominator. Every API-listed TWC event is included, with future listings marked explicitly.
- Quality is compared on the same 90 completed days per city. The union of affected events prevents counting the Miami bracket and exact mismatch twice; the three newer days are marked unaudited, rather than assigned zero issues.
- Terminal open interest is reported separately because the schema does not guarantee a comparable historical measure. The acquisition receipt times make the remaining unsettled OI values identifiable as observations during a request window.

Verified ignored exact-body cache: 66 responses; 2264465 compressed bytes.
This run: 0 new network responses; 162 settlement-cache reads.
The five failure demonstrations restored every original function; final checks:


PASS: Every event forms an integer partition or lists every gap, overlap and missing tail.
PASS: The climate day matches the directive and real CLI products at both DST changes.
PASS: Rankings account for every TWC event and every missing volume or OI field.
PASS: Quality resolves twenty-percent volume ties without using terminal open interest.


**Out-of-scope findings, left unchanged:**

- Canonical KX series queries also serve older HIGH-family events, including `HIGHCHI-2` metadata. These are counted as excluded legacy-family inventory, rather than mixed into the requested KX partitions or TWC ranking.
- The TWC intraday aggregation window is not specified by the examined KX primary rules; no TWC clock-boundary equivalence is asserted or implemented.
- The separate API-history/rate-limit spike remains open; only the documentation needed for these research requests was checked. Production collectors, schemas, pricing, plan and decisions remain unchanged.